# Investment Research Agent — End-to-End Integration Notebook

This notebook is the **integration and demonstration layer** for the AAI-520 Investment Research Agent.

The notebook starts at **Initialize Session**. Project background, environment setup, package installation, and detailed architecture discussion belong in the accompanying report.

### Integration boundary

- **Cohort 1:** supplies research/company/market data through an adapter.
- **Cohort 2:** owns RAG + LLM generation, prompt chaining, and routing. Its Router Service produces the **initial response**.
- **Cohort 3:** `MemoryAwareResearchService` owns memory retrieval, evaluator–optimizer execution, self-reflection/optimization, and persistence.

The critical flow demonstrated here is:

`Question → Cohort 2 Router → Initial Response → Cohort 3 Evaluation/Optimization → Final Response → Memory`

## 1. Initialize Session

In [ ]:
from dataclasses import dataclass, field
from datetime import datetime, timezone
from uuid import uuid4

SESSION_ID = f"session-{uuid4().hex[:10]}"
SESSION_STARTED_AT = datetime.now(timezone.utc).isoformat()

print(f"Session: {SESSION_ID}")
print(f"Started: {SESSION_STARTED_AT}")


## 2. User Input — Stock Symbol

Enter the company/security to research. The symbol is retained for all subsequent questions in this session.

In [ ]:
stock_symbol = input("Enter stock symbol (e.g., AAPL): ").strip().upper()

if not stock_symbol:
    raise ValueError("A stock symbol is required.")

print(f"Research symbol: {stock_symbol}")


## 3. Cohort 1 — Research Data Adapter

Cohort 1 owns data acquisition. The integration notebook should consume its output rather than duplicate its implementation.

The adapter below is intentionally a small contract/stub so the notebook remains runnable while the actual Cohort 1 implementation is wired in.

In [ ]:
class Cohort1ResearchDataAdapter:
    """Adapter boundary for Cohort 1 data acquisition."""

    def collect(self, symbol: str) -> dict:
        """
        Replace this body with the Cohort 1 implementation.

        Expected output should contain the financial/company/news data
        required by Cohort 2 for RAG and LLM generation.
        """
        return {
            "symbol": symbol,
            "sources": [],
            "documents": [],
            "financial_data": {},
            "status": "COHORT_1_PLACEHOLDER",
        }


cohort1 = Cohort1ResearchDataAdapter()
research_data = cohort1.collect(stock_symbol)

print(f"Cohort 1 status: {research_data['status']}")


## 4. Agent Function #1 — Research Planning

The planner determines the research steps for the selected company/question. If Cohort 1/2 already exposes a planner, use that implementation here rather than creating another planning engine.

In [ ]:
def build_research_plan(symbol: str) -> list[str]:
    """Integration-level representation of the research plan."""
    return [
        f"Collect current and historical information for {symbol}",
        "Retrieve relevant company/news/financial evidence",
        "Route the question to the appropriate Cohort 2 specialist",
        "Generate an evidence-grounded initial response",
        "Evaluate and optimize the response through Cohort 3",
        "Persist useful learning for subsequent questions",
    ]


research_plan = build_research_plan(stock_symbol)
for step_number, step in enumerate(research_plan, start=1):
    print(f"{step_number}. {step}")


## 5. Workflow 1 — Prompt Chaining

The required chain is:

`Ingest News → Preprocess → Classify → Extract → Summarize`

The actual Cohort 2 implementation should be invoked here. The notebook only displays the workflow boundary.

In [ ]:
prompt_chain_stages = [
    "Ingest News",
    "Preprocess",
    "Classify",
    "Extract",
    "Summarize",
]

print(" → ".join(prompt_chain_stages))

# TODO: replace this display-only cell with the Cohort 2 prompt-chain
# implementation when its public interface is wired into the project.


## 6. Workflow 2 — Routing and Initial Response

**This is the key integration correction.**

The initial response is generated by the **Cohort 2 Router Service**. `MemoryAwareResearchService` no longer accepts an externally generated `answer`.

The Cohort 3 boundary is therefore:

`router_service.route(query, context) → initial_response → MemoryAwareResearchService.run(...)`

In [ ]:
from evaluator_optimizer.orchestration.memory_aware_orchestrator import (
    MemoryAwareResearchService,
)

# IMPORTANT:
# Import/construct the real Cohort 2 Router Service here.
# The amended Cohort 3 service expects the following contract:
#
#     router_service.route(query: str, context: str) -> str
#
# Example project-specific wiring (replace with the actual Cohort 2 class):
# from <cohort2_module> import RouterService
# router_service = RouterService(...)

router_service = None  # Replace with the actual Cohort 2 Router Service.

if router_service is None:
    print("Cohort 2 Router Service is a required integration dependency.")
    print("Wire the actual Cohort 2 implementation before executing the end-to-end run.")


## 7. Cohort 3 Service Construction

`MemoryAwareResearchService` is the integration-facing Cohort 3 facade. Its internal evaluator, reflection, optimization, and memory components remain encapsulated.

In [ ]:
# Wire these objects using the existing Cohort 3 construction/configuration.
# Do not recreate their implementation in this notebook.

# from evaluator_optimizer.memory.memory_service import MemoryService
# from evaluator_optimizer.memory.memory_builder import MemoryBuilder
# from evaluator_optimizer.optimization.optimization_controller import OptimizationController
#
# memory_service = ...
# memory_builder = ...
# optimization_controller = ...
#
# research_service = MemoryAwareResearchService(
#     router_service=router_service,
#     memory_service=memory_service,
#     memory_builder=memory_builder,
#     optimization_controller=optimization_controller,
# )

research_service = None  # Replace with the fully configured Cohort 3 service.


## 8. Initial Company Research Report

Use the actual Cohort 1/Cohort 2 implementation to build the research/RAG context. This notebook deliberately does not duplicate financial-data acquisition or RAG implementation.

In [ ]:
def build_research_context(data: dict) -> str:
    """Convert Cohort 1 output into the context expected by Cohort 2."""
    # Replace with the actual Cohort 2 context-construction/RAG output.
    return str(data)


research_context = build_research_context(research_data)
print(research_context[:2000])


## 9. User Follow-up Questions

The session supports repeated questions. Each question goes independently through the Cohort 2 → Cohort 3 boundary while the Cohort 3 memory layer can reuse prior learning.

In [ ]:
questions = []
for index in range(5):
    question = input(
        f"Follow-up question {index + 1} "
        "(press Enter to stop): "
    ).strip()
    if not question:
        break
    questions.append(question)

print(f"Questions captured: {len(questions)}")


## 10. Execute Questions — Cohort 2 → Cohort 3

For every question:

1. `MemoryAwareResearchService` retrieves relevant memory.
2. Cohort 2 Router generates the **initial response**.
3. Cohort 3 evaluates that response.
4. Reflection/optimization is performed through the existing `OptimizationController`.
5. The final response is persisted as memory.
6. The returned `EvaluationTrace` provides evidence for the notebook metrics.

In [ ]:
question_results = []

if research_service is None:
    print("Execution skipped: configure research_service in the previous cell.")
else:
    for question_number, question in enumerate(questions, start=1):
        trace, memory_context = research_service.run(
            exchange_symbol=stock_symbol,
            query=question,
            context=research_context,
            topics=[],
        )

        question_results.append(
            {
                "question_number": question_number,
                "question": question,
                "trace": trace,
                "memory_context": memory_context,
            }
        )

    print(f"Completed questions: {len(question_results)}")


## 11. Workflow 3 — Evaluator–Optimizer Trace

The notebook reports the trace produced by Cohort 3 rather than reproducing the evaluation/optimization algorithm.

In [ ]:
def trace_summary(result: dict) -> dict:
    """Extract presentation-level trace information defensively."""
    trace = result["trace"]
    return {
        "question_number": result["question_number"],
        "question": result["question"],
        "initial_response": getattr(trace, "initial_answer", None),
        "final_response": (
            getattr(trace, "optimized_answer", None)
            or getattr(trace, "initial_answer", None)
        ),
        "initial_evaluation": getattr(trace, "initial_evaluation", None),
        "final_evaluation": getattr(trace, "final_evaluation", None),
        "memory_retrieved": getattr(trace, "memory_retrieved", []),
        "memory_applied": getattr(trace, "memory_applied", []),
    }


trace_rows = [trace_summary(item) for item in question_results]
trace_rows


## 12. Agent Function #3 — Self-Reflection

Self-reflection is demonstrated by the evaluation/reflection information contained in each `EvaluationTrace`. The notebook presents those outputs for traceability.

In [ ]:
for row in trace_rows:
    print(f"\nQuestion {row['question_number']}: {row['question']}")
    print("Initial evaluation:", row["initial_evaluation"])
    print("Final evaluation:", row["final_evaluation"])
    print("Memory retrieved:", row["memory_retrieved"])


## 13. Agent Function #4 — Learning Across Runs

Cohort 3 persists the accepted response as memory. Subsequent similar questions can retrieve those memories through `MemoryAwareResearchService.retrieve_memory()`.

In [ ]:
for row in trace_rows:
    print(
        f"Question {row['question_number']}: "
        f"retrieved={row['memory_retrieved']} | "
        f"applied={row['memory_applied']}"
    )


## 14. Final Responses

Display the optimized response for each follow-up question.

In [ ]:
for row in trace_rows:
    print("=" * 80)
    print(f"Question {row['question_number']}: {row['question']}")
    print("\nFinal response:\n")
    print(row["final_response"])


## 15. Final Traceability Table

Only metrics actually emitted by the Cohort 3 trace should be populated. Do not fabricate scores or iteration counts in the integration report.

In [ ]:
import pandas as pd

traceability_df = pd.DataFrame(
    [
        {
            "Question": row["question"],
            "Initial Response": row["initial_response"],
            "Initial Evaluation": row["initial_evaluation"],
            "Final Evaluation": row["final_evaluation"],
            "Memory Retrieved": row["memory_retrieved"],
            "Memory Applied": row["memory_applied"],
            "Final Response": row["final_response"],
        }
        for row in trace_rows
    ]
)

display(traceability_df)


## 16. Evaluation Improvement Visualization

This section should be populated from the concrete numeric evaluation fields exposed by `EvaluationTrace`. The code below intentionally avoids assuming field names that have not been verified.

In [ ]:
# Example extraction hook once the concrete EvaluationTrace metric fields
# are confirmed in the project source:
#
# metrics_df = pd.DataFrame(...)
# metrics_df.plot(x="question", y=["initial_score", "final_score"], kind="bar")
# plt.title("Evaluation Improvement")
# plt.ylabel("Score")
# plt.show()

print("Populate visualization from actual EvaluationTrace metrics only.")


## 17. End-to-End Workflow Summary

The implemented integration demonstrates the four autonomous functions and three workflow patterns without duplicating cohort implementation logic:

| Requirement | Integration boundary |
|---|---|
| Planning | Research plan / Cohort planning component |
| Dynamic tool/data use | Cohort 1 data adapter + Cohort 2 RAG/tooling |
| Self-reflection | Cohort 3 evaluation/reflection trace |
| Learning across runs | Cohort 3 memory retrieval and persistence |
| Prompt Chaining | Cohort 2 |
| Routing | Cohort 2 Router Service |
| Evaluator–Optimizer | Cohort 3 `MemoryAwareResearchService` + `OptimizationController` |

### Critical hand-off

`Cohort 2 Router Service → initial_response → MemoryAwareResearchService → EvaluationTrace → final response + memory`